# slicerator

## Example 1: wrap a sequence and slice it lazily

In [ ]:
# Example 1: slicing a Slicerator returns another Slicerator; nothing is read until it is iterated
from slicerator import Slicerator

reads = []
class Recorder(list):
    def __getitem__(self, i):
        reads.append(i)
        return super().__getitem__(i)

s = Slicerator(Recorder(range(10)))
evens = s[::2]
assert isinstance(evens, Slicerator) and reads == []
assert list(evens) == [0, 2, 4, 6, 8] and reads == [0, 2, 4, 6, 8]
print("Example 1:", len(s), "items, evens", list(evens))

## Example 2: chained slices compose

In [ ]:
# Example 2: a slice of a slice maps back to the original indices
from slicerator import Slicerator

s = Slicerator(list("abcdefghij"))
part = s[2:8][::2]
assert list(part) == ["c", "e", "g"] and len(part) == 3
assert list(s[::-1][:3]) == ["j", "i", "h"]
print("Example 2:", list(part), list(s[::-1][:3]))

## Example 3: fancy indexing with a list or a boolean mask

In [ ]:
# Example 3: a list of positions and a boolean mask both select items
import numpy as np
from slicerator import Slicerator

s = Slicerator(list(range(100, 110)))
assert list(s[[0, 3, 9]]) == [100, 103, 109]
mask = np.arange(10) % 3 == 0
assert list(s[mask]) == [100, 103, 106, 109]
print("Example 3:", list(s[[0, 3, 9]]), list(s[mask]))

## Example 4: pipeline maps a function lazily over a Slicerator

In [ ]:
# Example 4: a @pipeline function applied to a Slicerator is evaluated only for the items you read
from slicerator import Slicerator, pipeline

seen = []

@pipeline
def square(x):
    seen.append(x)
    return x * x

squares = square(Slicerator(list(range(10))))
assert seen == []
assert list(squares[3:6]) == [9, 16, 25] and seen == [3, 4, 5]
assert square(7) == 49  # on a plain value it is just the function
print("Example 4:", list(squares[3:6]), "computed for", seen)

## Example 5: a Slicerator from a function and a length

In [ ]:
# Example 5: Slicerator.from_func turns any index -> value function into a sliceable sequence
from slicerator import Slicerator

cubes = Slicerator.from_func(lambda i: i ** 3, length=20)
assert len(cubes) == 20 and cubes[3] == 27
assert list(cubes[-3:]) == [17 ** 3, 18 ** 3, 19 ** 3]
print("Example 5:", cubes[3], list(cubes[-3:]))